# 02 · Feature Engineering

**Goal:** Transform raw transactions into one row per customer
with ML-ready features.

| Feature | Description | Why it matters |
|---------|-------------|----------------|
| Recency | Days since last purchase | Recent buyers are more valuable |
| Frequency | # distinct invoices | Habit formation signal |
| Monetary | Total spend £ | Direct value indicator |
| AvgBasket | Mean spend per order | Quality-of-purchase signal |
| TotalItems | Units purchased | Engagement breadth |
| Log* | Log-transformed variants | Reduces skew for ML |

## 0 · Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.data_loader import load_and_clean
from src.features    import build_rfm, add_churn_label, get_feature_sets

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "axes.titleweight": "bold"})
PALETTE = ["#4361EE", "#3A0CA3", "#7209B7", "#F72585", "#4CC9F0"]
sns.set_theme(style="whitegrid", palette=PALETTE)


## 1 · Load clean data

In [ ]:
df  = load_and_clean("../data/Online_Retail.xlsx")
rfm = build_rfm(df)
rfm.head()


## 2 · RFM distributions — raw vs log

In [ ]:
features_raw = ["Recency", "Frequency", "Monetary"]
features_log = ["Recency", "Frequency", "LogMonetary"]   # Recency stays linear

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("RFM Distributions: Raw (top) vs Log-transformed (bottom)", fontsize=13)

for i, col in enumerate(features_raw):
    axes[0, i].hist(rfm[col], bins=50, color=PALETTE[i], edgecolor="white", lw=0.3)
    axes[0, i].set_title(f"{col} — raw")
    skew = stats.skew(rfm[col])
    axes[0, i].text(0.97, 0.95, f"skew={skew:.2f}",
                    transform=axes[0, i].transAxes, ha="right", va="top", fontsize=9)

for i, col in enumerate(features_log):
    axes[1, i].hist(rfm[col], bins=50, color=PALETTE[i], edgecolor="white", lw=0.3)
    axes[1, i].set_title(f"{col}")
    skew = stats.skew(rfm[col])
    axes[1, i].text(0.97, 0.95, f"skew={skew:.2f}",
                    transform=axes[1, i].transAxes, ha="right", va="top", fontsize=9)

plt.tight_layout()
plt.savefig("../reports/figures/feat_rfm_distributions.png", bbox_inches="tight")
plt.show()


## 3 · Outlier inspection

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, col in enumerate(["Recency", "Frequency", "Monetary"]):
    axes[i].boxplot(rfm[col], vert=True, patch_artist=True,
                    boxprops=dict(facecolor=PALETTE[i], alpha=0.7))
    axes[i].set_title(f"{col}")
    q99 = rfm[col].quantile(0.99)
    axes[i].set_ylim(0, q99 * 1.1)
plt.suptitle("Outlier Inspection (99th pct cap)", fontsize=12)
plt.tight_layout()
plt.savefig("../reports/figures/feat_outliers.png", bbox_inches="tight")
plt.show()

print("99th percentile values:")
print(rfm[["Recency","Frequency","Monetary"]].quantile(0.99).round(2))


## 4 · Feature correlation matrix

In [ ]:
feat_cols = ["Recency","Frequency","Monetary","AvgBasket","TotalItems"]
corr = rfm[feat_cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            linewidths=0.5, ax=ax, mask=mask, vmin=-1, vmax=1)
ax.set_title("RFM Feature Correlation")
plt.tight_layout()
plt.savefig("../reports/figures/feat_correlation.png", bbox_inches="tight")
plt.show()


## 5 · Add churn label

In [ ]:
rfm = add_churn_label(rfm, threshold_days=90)

fig, ax = plt.subplots(figsize=(5, 4))
counts = rfm["Churned"].value_counts()
ax.bar(["Active (0)", "Churned (1)"], counts.values,
       color=[PALETTE[0], "#e63946"])
for i, v in enumerate(counts.values):
    ax.text(i, v + 30, f"{v:,}\n({v/len(rfm)*100:.1f}%)", ha="center", fontsize=10)
ax.set_title("Churn Label Distribution")
ax.set_ylabel("# Customers")
plt.tight_layout()
plt.savefig("../reports/figures/feat_churn_balance.png", bbox_inches="tight")
plt.show()


## 6 · RFM by churn status

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, col in enumerate(["Recency", "Frequency", "LogMonetary"]):
    for label, grp in rfm.groupby("Churned"):
        axes[i].hist(grp[col], bins=40, alpha=0.55,
                     label=f"{'Churned' if label else 'Active'}",
                     color=["#e63946", PALETTE[0]][label == 0])
    axes[i].set_title(f"{col} by Churn Status")
    axes[i].legend()
plt.tight_layout()
plt.savefig("../reports/figures/feat_rfm_by_churn.png", bbox_inches="tight")
plt.show()


## 7 · Feature set definitions

In [ ]:
feat_sets = get_feature_sets()
for name, cols in feat_sets.items():
    print(f"  {name:<15}: {cols}")


## 8 · Save processed data

In [ ]:
os.makedirs("../data/processed", exist_ok=True)
rfm.to_csv("../data/processed/rfm_features.csv", index=False)
print(f"Saved {len(rfm):,} rows → ../data/processed/rfm_features.csv")
rfm.describe().round(2)


## Summary

- Log-transforming Monetary, AvgBasket, and TotalItems **significantly reduces skew**.
- Recency and Monetary are **negatively correlated** — high spenders buy more recently.
- Frequency and Monetary are **highly correlated** (0.78) — frequent buyers spend more.
- The churn label (>90 days inactive) creates a **33/67 class imbalance** — consider
  class_weight='balanced' in classifiers.